## Import the libraries

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import zscore
%pip install gtts
from gtts import gTTS

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 8.0 MB/s eta 0:00:00
  Attempting uninstall: click
    Found existing installation: click 8.3.1
    Uninstalling click-8.3.1:
      Successfully uninstalled click-8.3.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
typer 0.24.1 requires click>=8.2.1, but you have click 8.1.8 which is incompatible.


In [ ]:
import kagglehub
# Download latest version
path = kagglehub.dataset_download("vivek468/superstore-dataset-final")
print("Path to dataset files:", path)

Using Colab cache for faster access to the 'superstore-dataset-final' dataset.
Path to dataset files: /kaggle/input/superstore-dataset-final


In [ ]:
import os
file_path = os.path.join(path, "Sample - Superstore.csv")
df = pd.read_csv(file_path, encoding='latin1')
print(df.head())

   Row ID        Order ID  Order Date   Ship Date       Ship Mode Customer ID  \
0       1  CA-2016-152156   11/8/2016  11/11/2016    Second Class    CG-12520   
1       2  CA-2016-152156   11/8/2016  11/11/2016    Second Class    CG-12520   
2       3  CA-2016-138688   6/12/2016   6/16/2016    Second Class    DV-13045   
3       4  US-2015-108966  10/11/2015  10/18/2015  Standard Class    SO-20335   
4       5  US-2015-108966  10/11/2015  10/18/2015  Standard Class    SO-20335   

     Customer Name    Segment        Country             City  ...  \
0      Claire Gute   Consumer  United States        Henderson  ...   
1      Claire Gute   Consumer  United States        Henderson  ...   
2  Darrin Van Huff  Corporate  United States      Los Angeles  ...   
3   Sean O'Donnell   Consumer  United States  Fort Lauderdale  ...   
4   Sean O'Donnell   Consumer  United States  Fort Lauderdale  ...   

  Postal Code  Region       Product ID         Category Sub-Category  \
0       42420   Sout

In [ ]:
df['Order Date'] = pd.to_datetime(df['Order Date'])
print(df.head())

   Row ID        Order ID Order Date   Ship Date       Ship Mode Customer ID  \
0       1  CA-2016-152156 2016-11-08  11/11/2016    Second Class    CG-12520   
1       2  CA-2016-152156 2016-11-08  11/11/2016    Second Class    CG-12520   
2       3  CA-2016-138688 2016-06-12   6/16/2016    Second Class    DV-13045   
3       4  US-2015-108966 2015-10-11  10/18/2015  Standard Class    SO-20335   
4       5  US-2015-108966 2015-10-11  10/18/2015  Standard Class    SO-20335   

     Customer Name    Segment        Country             City  ...  \
0      Claire Gute   Consumer  United States        Henderson  ...   
1      Claire Gute   Consumer  United States        Henderson  ...   
2  Darrin Van Huff  Corporate  United States      Los Angeles  ...   
3   Sean O'Donnell   Consumer  United States  Fort Lauderdale  ...   
4   Sean O'Donnell   Consumer  United States  Fort Lauderdale  ...   

  Postal Code  Region       Product ID         Category Sub-Category  \
0       42420   South  FUR

In [ ]:
cube = df.groupby(["Region","Category"])["Sales"].sum().reset_index()

In [ ]:
cube

,Region,Category,Sales
0,Central,Furniture,163797.1638
1,Central,Office Supplies,167026.4150
2,Central,Technology,170416.3120
3,East,Furniture,208291.2040
4,East,Office Supplies,205516.0550
5,East,Technology,264973.9810
6,South,Furniture,117298.6840
7,South,Office Supplies,125651.3130
8,South,Technology,148771.9080
9,West,Furniture,252612.7435


##Insight Extraction

In [ ]:
insights = []

max_sales = cube["Sales"].max()

for _,row in cube.iterrows():

    score = row["Sales"]/max_sales

    insight = {
        "text":f"{row['Category']} in {row['Region']} has sales {row['Sales']}",
        "interestingness":score,
        "cost":5
    }

    insights.append(insight)

##Outlier Insights

In [ ]:
cube["z"] = zscore(cube["Sales"])

outliers = cube[abs(cube["z"])>1.5]

for _,row in outliers.iterrows():

    insights.append({
        "text":f"{row['Category']} in {row['Region']} shows unusual sales",
        "interestingness":abs(row["z"]),
        "cost":6
    })

In [ ]:
max_row = cube.loc[cube["Sales"].idxmax()]
min_row = cube.loc[cube["Sales"].idxmin()]

insights.append({
"text":f"{max_row['Category']} in {max_row['Region']} sells much more than {min_row['Category']} in {min_row['Region']}",
"interestingness":1.0,
"cost":7
})

##Remove Dominated Insights

In [ ]:
def remove_dominated(insights):

    filtered = []

    for i in insights:
        dominated = False

        for j in insights:

            if (j["cost"] <= i["cost"]) and (j["interestingness"] >= i["interestingness"]) and j!=i:
                dominated = True

        if not dominated:
            filtered.append(i)

    return filtered

insights = remove_dominated(insights)

In [ ]:
slopes = []

for i in range(len(insights)):

    for j in range(i+1,len(insights)):

        s1 = insights[i]
        s2 = insights[j]

        if s1["cost"] != s2["cost"]:

            slope = (s2["interestingness"] - s1["interestingness"]) / (s2["cost"] - s1["cost"])

            slopes.append(slope)

In [ ]:
## Compute Alpha
alpha = np.median(slopes)

##Score Insights Using Alpha

In [ ]:
##maximize int(s) − α × cost(s)

for i in insights:
    i["score"] = i["interestingness"] - alpha * i["cost"]

##Select Insights Under Time Budget

In [ ]:
time_budget = 30

insights = sorted(insights,key=lambda x:x["score"],reverse=True)

selected = []

time_used = 0

for i in insights:

    if time_used + i["cost"] <= time_budget:

        selected.append(i)
        time_used += i["cost"]

In [ ]:
text = ""

for i in selected:

    text += i["text"] + ". "

print(text)

Technology in East has sales 264973.981. Furniture in South shows unusual sales. 


In [ ]:
tts = gTTS(text)
tts.save("insight_voice.mp3")

In [ ]:
tts

In [ ]:
from IPython.display import Audio

Audio("insight_voice.mp3")